# Tokenization in Natural Language Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/nlp-learning-journey/blob/main/examples/tokenization.ipynb)

## Overview

Tokenization is the process of breaking down text into smaller units called tokens. These tokens can be words, subwords, characters, or other meaningful elements. It's one of the fundamental preprocessing steps in NLP.

## What You'll Learn

- Different types of tokenization
- Word-level tokenization
- Subword tokenization
- Sentence tokenization
- Using popular NLP libraries (NLTK, spaCy, Transformers)

## Prerequisites

Basic understanding of Python and text processing concepts.

## Setup and Installation

Let's install the required libraries for this notebook.

In [3]:
# Environment Detection and Setup
import sys
import subprocess

# Detect the runtime environment
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

print(f"Environment detected:")
print(f"  - Local: {IS_LOCAL}")
print(f"  - Google Colab: {IS_COLAB}")
print(f"  - Kaggle: {IS_KAGGLE}")

# Platform-specific system setup
if IS_COLAB:
    print("\nSetting up Google Colab environment...")
    !apt update -qq
    # Install system dependencies that might be needed for NLP
    !apt install -y -qq libpq-dev
elif IS_KAGGLE:
    print("\nSetting up Kaggle environment...")
    # Kaggle usually has most packages pre-installed
else:
    print("\nSetting up local environment...")

# Install required packages for this notebook
required_packages = [
    "nltk",
    "spacy",
    "transformers",
    "tokenizers",
    "pandas",
    "matplotlib",
    "seaborn"
]

print("\nInstalling required packages...")
for package in required_packages:
    if IS_COLAB or IS_KAGGLE:
        !pip install -q {package}
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package],
                      capture_output=True)
    print(f"✓ {package}")

# Download spaCy model
print("\nDownloading spaCy English model...")
try:
    if IS_COLAB or IS_KAGGLE:
        !python -m spacy download en_core_web_sm
    else:
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
                      capture_output=True)
    print("✓ spaCy model downloaded successfully")
except Exception as e:
    print(f"⚠️  spaCy model download failed: {e}")

print("\n🎉 Environment setup complete!")

Environment detected:
  - Local: False
  - Google Colab: True
  - Kaggle: False

Setting up Google Colab environment...
44 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
libpq-dev is already the newest version (14.19-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 44 not upgraded.

Installing required packages...
✓ nltk
✓ spacy
✓ transformers
✓ tokenizers
✓ pandas
✓ matplotlib
✓ seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart ke

In [4]:
# Import required libraries
import nltk
import spacy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
from tokenizers import Tokenizer
import re

# Download NLTK data with error handling
nltk_datasets = ['punkt', 'punkt_tab', 'wordnet', 'stopwords']
print("Downloading NLTK datasets...")
for dataset in nltk_datasets:
    try:
        nltk.download(dataset, quiet=True)
        print(f"✓ {dataset}")
    except Exception as e:
        print(f"⚠️  Failed to download {dataset}: {e}")

# Load spaCy model with error handling
try:
    nlp = spacy.load('en_core_web_sm')
    print("✓ spaCy model loaded successfully")
except OSError:
    print("⚠️  spaCy model not found. Please run the setup cell above.")
    nlp = None

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

✓ punkt
✓ punkt_tab
✓ wordnet
✓ stopwords
✓ spaCy model loaded successfully


## Sample Text

Let's use a sample text to demonstrate different tokenization techniques.

In [5]:
sample_text = """
G'day mate! Australia is a bonkers beautiful country with unique wildlife like kangaroos,
koalas, and wombats. From the stunning Sydney Harbour Bridge to the rugged Outback,
Aussie landscapes are fair dinkum spectacular! Crikey, let's explore some ripper
tokenization techniques on this Aussie text. Check out https://my.gov.au/ for more info!
"""

print("Sample Text:")
print(sample_text)

Sample Text:

G'day mate! Australia is a bonkers beautiful country with unique wildlife like kangaroos, 
koalas, and wombats. From the stunning Sydney Harbour Bridge to the rugged Outback, 
Aussie landscapes are fair dinkum spectacular! Crikey, let's explore some ripper 
tokenization techniques on this Aussie text. Check out https://my.gov.au/ for more info!



## 1. Basic Tokenization

### Simple Split-based Tokenization

The simplest form of tokenization is splitting text by whitespace.

In [6]:
# Basic whitespace tokenization
basic_tokens = sample_text.split()

print("Basic Split Tokenization:")
print(f"Number of tokens: {len(basic_tokens)}")
print("First 10 tokens:", basic_tokens[:10])
print("\nLimitations: Notice punctuation is attached to words")

Basic Split Tokenization:
Number of tokens: 49
First 10 tokens: ["G'day", 'mate!', 'Australia', 'is', 'a', 'bonkers', 'beautiful', 'country', 'with', 'unique']

Limitations: Notice punctuation is attached to words


### Regular Expression Tokenization

We can use regular expressions for more sophisticated tokenization.

In [7]:
# Regular expression tokenization
import re

# Pattern to match words (alphanumeric characters)
pattern = r'\b\w+\b'
regex_tokens = re.findall(pattern, sample_text)

print("Regular Expression Tokenization:")
print(f"Number of tokens: {len(regex_tokens)}")
print("First 10 tokens:", regex_tokens[:10])
print("\nNote: Punctuation is now separated from words")

Regular Expression Tokenization:
Number of tokens: 54
First 10 tokens: ['G', 'day', 'mate', 'Australia', 'is', 'a', 'bonkers', 'beautiful', 'country', 'with']

Note: Punctuation is now separated from words


## 2. NLTK Tokenization

NLTK provides robust tokenization methods that handle various edge cases.

In [8]:
# NLTK Word Tokenization
from nltk.tokenize import word_tokenize, sent_tokenize

nltk_word_tokens = word_tokenize(sample_text)

print("NLTK Word Tokenization:")
print(f"Number of tokens: {len(nltk_word_tokens)}")
print("All tokens:", nltk_word_tokens)
print("\nAdvantages: Handles punctuation, contractions, and special cases better")

NLTK Word Tokenization:
Number of tokens: 61
All tokens: ["G'day", 'mate', '!', 'Australia', 'is', 'a', 'bonkers', 'beautiful', 'country', 'with', 'unique', 'wildlife', 'like', 'kangaroos', ',', 'koalas', ',', 'and', 'wombats', '.', 'From', 'the', 'stunning', 'Sydney', 'Harbour', 'Bridge', 'to', 'the', 'rugged', 'Outback', ',', 'Aussie', 'landscapes', 'are', 'fair', 'dinkum', 'spectacular', '!', 'Crikey', ',', 'let', "'s", 'explore', 'some', 'ripper', 'tokenization', 'techniques', 'on', 'this', 'Aussie', 'text', '.', 'Check', 'out', 'https', ':', '//my.gov.au/', 'for', 'more', 'info', '!']

Advantages: Handles punctuation, contractions, and special cases better


In [9]:
# NLTK Sentence Tokenization
sentences = sent_tokenize(sample_text)

print("NLTK Sentence Tokenization:")
print(f"Number of sentences: {len(sentences)}")
for i, sent in enumerate(sentences, 1):
    print(f"Sentence {i}: {sent.strip()}")

NLTK Sentence Tokenization:
Number of sentences: 5
Sentence 1: G'day mate!
Sentence 2: Australia is a bonkers beautiful country with unique wildlife like kangaroos, 
koalas, and wombats.
Sentence 3: From the stunning Sydney Harbour Bridge to the rugged Outback, 
Aussie landscapes are fair dinkum spectacular!
Sentence 4: Crikey, let's explore some ripper 
tokenization techniques on this Aussie text.
Sentence 5: Check out https://my.gov.au/ for more info!


## 3. spaCy Tokenization

spaCy provides industrial-strength tokenization with linguistic awareness.

In [10]:
# spaCy tokenization
doc = nlp(sample_text)

print("spaCy Tokenization:")
print(f"Number of tokens: {len(doc)}")

# Display tokens with additional information
print("\nTokens with linguistic information:")
for token in doc[:15]:  # First 15 tokens
    print(f"Token: '{token.text}' | Lemma: '{token.lemma_}' | POS: {token.pos_} | Is_alpha: {token.is_alpha}")

spaCy Tokenization:
Number of tokens: 64

Tokens with linguistic information:
Token: '
' | Lemma: '
' | POS: SPACE | Is_alpha: False
Token: 'G'day' | Lemma: 'G'day' | POS: PROPN | Is_alpha: False
Token: 'mate' | Lemma: 'mate' | POS: NOUN | Is_alpha: True
Token: '!' | Lemma: '!' | POS: PUNCT | Is_alpha: False
Token: 'Australia' | Lemma: 'Australia' | POS: PROPN | Is_alpha: True
Token: 'is' | Lemma: 'be' | POS: AUX | Is_alpha: True
Token: 'a' | Lemma: 'a' | POS: DET | Is_alpha: True
Token: 'bonkers' | Lemma: 'bonker' | POS: NOUN | Is_alpha: True
Token: 'beautiful' | Lemma: 'beautiful' | POS: ADJ | Is_alpha: True
Token: 'country' | Lemma: 'country' | POS: NOUN | Is_alpha: True
Token: 'with' | Lemma: 'with' | POS: ADP | Is_alpha: True
Token: 'unique' | Lemma: 'unique' | POS: ADJ | Is_alpha: True
Token: 'wildlife' | Lemma: 'wildlife' | POS: NOUN | Is_alpha: True
Token: 'like' | Lemma: 'like' | POS: ADP | Is_alpha: True
Token: 'kangaroos' | Lemma: 'kangaroo' | POS: NOUN | Is_alpha: True


In [11]:
# spaCy sentence segmentation
print("spaCy Sentence Segmentation:")
for i, sent in enumerate(doc.sents, 1):
    print(f"Sentence {i}: {sent.text.strip()}")

spaCy Sentence Segmentation:
Sentence 1: G'day mate!
Sentence 2: Australia is a bonkers beautiful country with unique wildlife like kangaroos, 
koalas, and wombats.
Sentence 3: From the stunning Sydney Harbour Bridge to the rugged Outback, 
Aussie landscapes are fair dinkum spectacular!
Sentence 4: Crikey, let's explore some ripper 
tokenization techniques on this Aussie text.
Sentence 5: Check out https://my.gov.au/ for more info!


## 4. Subword Tokenization

Subword tokenization is crucial for handling out-of-vocabulary words and is widely used in modern NLP models.

### BERT Tokenization (WordPiece)

BERT uses WordPiece tokenization, which breaks words into subword units.

In [12]:
# BERT tokenizer (WordPiece)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the text
bert_tokens = bert_tokenizer.tokenize(sample_text)
bert_token_ids = bert_tokenizer.encode(sample_text)

print("BERT WordPiece Tokenization:")
print(f"Number of tokens: {len(bert_tokens)}")
print("\nFirst 20 tokens:")
for token in bert_tokens[:20]:
    print(f"'{token}'")

print("\nSpecial tokens:")
print(f"[CLS] token ID: {bert_tokenizer.cls_token_id}")
print(f"[SEP] token ID: {bert_tokenizer.sep_token_id}")
print(f"[PAD] token ID: {bert_tokenizer.pad_token_id}")

BERT WordPiece Tokenization:
Number of tokens: 85

First 20 tokens:
'g'
'''
'day'
'mate'
'!'
'australia'
'is'
'a'
'bon'
'##kers'
'beautiful'
'country'
'with'
'unique'
'wildlife'
'like'
'kangaroo'
'##s'
','
'ko'

Special tokens:
[CLS] token ID: 101
[SEP] token ID: 102
[PAD] token ID: 0


### GPT Tokenization (Byte Pair Encoding)

GPT models use Byte Pair Encoding (BPE) for tokenization.

In [13]:
# GPT tokenizer (BPE)
gpt_tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Set pad token (GPT-2 doesn't have one by default)
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

# Tokenize the text
gpt_tokens = gpt_tokenizer.tokenize(sample_text)
gpt_token_ids = gpt_tokenizer.encode(sample_text)

print("GPT BPE Tokenization:")
print(f"Number of tokens: {len(gpt_tokens)}")
print("\nFirst 20 tokens:")
for token in gpt_tokens[:20]:
    print(f"'{token}'")

print("\nNote: Ġ symbol represents beginning of a word in GPT tokenization")

GPT BPE Tokenization:
Number of tokens: 92

First 20 tokens:
'Ċ'
'G'
''d'
'ay'
'Ġmate'
'!'
'ĠAustralia'
'Ġis'
'Ġa'
'Ġbon'
'kers'
'Ġbeautiful'
'Ġcountry'
'Ġwith'
'Ġunique'
'Ġwildlife'
'Ġlike'
'Ġk'
'ang'
'aro'

Note: Ġ symbol represents beginning of a word in GPT tokenization


## 5. Tokenization Comparison

Let's compare the different tokenization methods on a complex example.

In [14]:
complex_text = "The pre-trained transformer-based models can handle out-of-vocabulary words like 'supercalifragilisticexpialidocious' effectively!"

print("Complex Text:", complex_text)
print("\n" + "="*50)

# Basic split
basic = complex_text.split()
print(f"\nBasic Split ({len(basic)} tokens):")
print(basic)

# NLTK
nltk_tokens = word_tokenize(complex_text)
print(f"\nNLTK ({len(nltk_tokens)} tokens):")
print(nltk_tokens)

# spaCy
spacy_doc = nlp(complex_text)
spacy_tokens = [token.text for token in spacy_doc]
print(f"\nspaCy ({len(spacy_tokens)} tokens):")
print(spacy_tokens)

# BERT
bert_tokens_complex = bert_tokenizer.tokenize(complex_text)
print(f"\nBERT WordPiece ({len(bert_tokens_complex)} tokens):")
print(bert_tokens_complex)

# GPT
gpt_tokens_complex = gpt_tokenizer.tokenize(complex_text)
print(f"\nGPT BPE ({len(gpt_tokens_complex)} tokens):")
print(gpt_tokens_complex)

Complex Text: The pre-trained transformer-based models can handle out-of-vocabulary words like 'supercalifragilisticexpialidocious' effectively!


Basic Split (11 tokens):
['The', 'pre-trained', 'transformer-based', 'models', 'can', 'handle', 'out-of-vocabulary', 'words', 'like', "'supercalifragilisticexpialidocious'", 'effectively!']

NLTK (13 tokens):
['The', 'pre-trained', 'transformer-based', 'models', 'can', 'handle', 'out-of-vocabulary', 'words', 'like', "'supercalifragilisticexpialidocious", "'", 'effectively', '!']

spaCy (22 tokens):
['The', 'pre', '-', 'trained', 'transformer', '-', 'based', 'models', 'can', 'handle', 'out', '-', 'of', '-', 'vocabulary', 'words', 'like', "'", 'supercalifragilisticexpialidocious', "'", 'effectively', '!']

BERT WordPiece (33 tokens):
['the', 'pre', '-', 'trained', 'transform', '##er', '-', 'based', 'models', 'can', 'handle', 'out', '-', 'of', '-', 'vocabulary', 'words', 'like', "'", 'super', '##cal', '##if', '##rag', '##ilis', '##tic', '##ex',

## 6. Practical Considerations

### Handling Special Cases

In [15]:
special_text = """
Email: user@example.com
URL: https://www.example.com/path?param=value
Phone: +1-555-123-4567
Hashtag: #NLP #MachineLearning
Mention: @username
Contraction: don't, won't, can't
Numbers: 3.14, $100.50, 1,000,000
Unicode: 😀 🌟 ñáéíóúあ好
"""

print("Special Cases Text:")
print(special_text)

print("\n" + "="*50)
print("\nNLTK Tokenization:")
nltk_special = word_tokenize(special_text)
print(nltk_special)

print("\nspaCy Tokenization:")
spacy_special = [token.text for token in nlp(special_text)]
print(spacy_special)

Special Cases Text:

Email: user@example.com
URL: https://www.example.com/path?param=value
Phone: +1-555-123-4567
Hashtag: #NLP #MachineLearning
Mention: @username
Contraction: don't, won't, can't
Numbers: 3.14, $100.50, 1,000,000
Unicode: 😀 🌟 ñáéíóúあ好



NLTK Tokenization:
['Email', ':', 'user', '@', 'example.com', 'URL', ':', 'https', ':', '//www.example.com/path', '?', 'param=value', 'Phone', ':', '+1-555-123-4567', 'Hashtag', ':', '#', 'NLP', '#', 'MachineLearning', 'Mention', ':', '@', 'username', 'Contraction', ':', 'do', "n't", ',', 'wo', "n't", ',', 'ca', "n't", 'Numbers', ':', '3.14', ',', '$', '100.50', ',', '1,000,000', 'Unicode', ':', '😀', '🌟', 'ñáéíóúあ好']

spaCy Tokenization:
['\n', 'Email', ':', 'user@example.com', '\n', 'URL', ':', 'https://www.example.com/path?param=value', '\n', 'Phone', ':', '+1', '-', '555', '-', '123', '-', '4567', '\n', 'Hashtag', ':', '#', 'NLP', '#', 'MachineLearning', '\n', 'Mention', ':', '@username', '\n', 'Contraction', ':', 'do', "n't", ',',

### Custom Tokenization Rules

In [16]:
def custom_tokenizer(text):
    """
    Custom tokenizer that preserves certain patterns
    """
    # Preserve emails, URLs, hashtags, mentions
    import re

    # Define patterns
    patterns = {
        'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        'url': r'https?://[^\s]+',
        'hashtag': r'#\w+',
        'mention': r'@\w+',
        'phone': r'\+?1?-?\(?\d{3}\)?-?\d{3}-?\d{4}',
    }

    tokens = []
    remaining_text = text

    # Find and extract special patterns
    for pattern_name, pattern in patterns.items():
        matches = re.finditer(pattern, remaining_text)
        for match in matches:
            tokens.append((match.group(), pattern_name))

    # Use NLTK for the rest
    for pattern in patterns.values():
        remaining_text = re.sub(pattern, ' ', remaining_text)

    regular_tokens = [(token, 'word') for token in word_tokenize(remaining_text) if token.strip()]

    return tokens + regular_tokens

# Test custom tokenizer
custom_tokens = custom_tokenizer(special_text)
print("Custom Tokenization with Type Labels:")
for token, token_type in custom_tokens[:20]:  # First 20 tokens
    print(f"'{token}' ({token_type})")

Custom Tokenization with Type Labels:
'user@example.com' (email)
'https://www.example.com/path?param=value' (url)
'#NLP' (hashtag)
'#MachineLearning' (hashtag)
'@example' (mention)
'@username' (mention)
'+1-555-123-4567' (phone)
'Email' (word)
':' (word)
'URL' (word)
':' (word)
'Phone' (word)
':' (word)
'Hashtag' (word)
':' (word)
'Mention' (word)
':' (word)
'Contraction' (word)
':' (word)
'do' (word)


## 7. Exercises

Try these exercises to practice tokenization:

### Exercise 1: Multilingual Tokenization

Test different tokenizers on multilingual text.

In [17]:
multilingual_text = """
English: Hello, how are you?
Vietnamese: Xin chào, bạn khỏe không?
Spanish: Hola, ¿cómo estás?
French: Bonjour, comment allez-vous?
German: Hallo, wie geht es dir?
Chinese: 你好，你好吗？
Japanese: こんにちは、元気ですか？
"""

print("Multilingual Text Tokenization:")
print("Original text:")
print(multilingual_text)

# TODO: Try different tokenizers on this multilingual text
# Compare NLTK, spaCy, and transformer tokenizers

# Your code here:


Multilingual Text Tokenization:
Original text:

English: Hello, how are you?
Vietnamese: Xin chào, bạn khỏe không?
Spanish: Hola, ¿cómo estás?
French: Bonjour, comment allez-vous?
German: Hallo, wie geht es dir?
Chinese: 你好，你好吗？
Japanese: こんにちは、元気ですか？



### Exercise 2: Token Statistics

Analyze tokenization statistics for a longer text.

In [18]:
# TODO: Implement a function that takes a text and returns:
# - Total number of tokens
# - Average token length
# - Most common tokens
# - Vocabulary size (unique tokens)

def analyze_tokenization(text, tokenizer_func):
    """
    Analyze tokenization statistics
    """
    # Your implementation here
    pass

# Test with a longer text (you can use any text you like)
long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language, in particular how to program computers to process and analyze
large amounts of natural language data. The goal is a computer capable of
understanding the contents of documents, including the contextual nuances of
the language within them.
"""

# Your analysis code here:


## Key Takeaways

1. **Different tokenizers for different purposes**:
   - Basic split: Fast but limited
   - NLTK: Good for traditional NLP tasks
   - spaCy: Industrial-strength with linguistic features
   - Transformer tokenizers: Essential for modern NLP models

2. **Subword tokenization advantages**:
   - Handles out-of-vocabulary words
   - More efficient vocabulary usage
   - Better for morphologically rich languages

3. **Consider your use case**:
   - Text preprocessing: NLTK or spaCy
   - Modern NLP models: Use matching tokenizer
   - Custom applications: May need custom rules

4. **Important considerations**:
   - Language support
   - Special character handling
   - Performance requirements
   - Downstream task compatibility

## Next Steps

- Explore text normalization techniques
- Learn about stemming and lemmatization
- Study different subword algorithms (BPE, WordPiece, SentencePiece)
- Practice with domain-specific texts (social media, biomedical, legal)

## Resources

- [NLTK Documentation](https://www.nltk.org/)
- [spaCy Documentation](https://spacy.io/)
- [Hugging Face Tokenizers](https://huggingface.co/docs/tokenizers/)
- [Neural Machine Translation of Rare Words with Subword Units](https://arxiv.org/abs/1508.07909)
